# LOTA training walkthrough

Cell-by-cell version of `train.py`, so each stage (config, preprocessing/data, model init, one training epoch, validation) can be run and re-run independently.

Run this notebook from the repo root (same directory as `train.py`) so the relative imports and `../GenImage_root` path resolve exactly as they do for the CLI script.

## Setup

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import types
import torch
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import util as toolkit
from loader import get_loader as fetch_train_data, get_val_loader as fetch_val_data, MODEL_NAME_MAP
from model import model as NeuralNetwork
from util import bceLoss as compute_binary_loss

print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## Config

Same fields/defaults as `config.py`, but built by hand as a `SimpleNamespace` instead of via `argparse` (argparse would try to parse Jupyter's own `sys.argv`). Edit values here directly instead of passing CLI flags.

This mirrors: `python train.py --choice 0 0 0 0 0 0 1 0 --image_root ../GenImage_root --bit_mode thresholding --patch_size 32 --patch_mode max`

In [ ]:
config = types.SimpleNamespace(
    batchsize=64,
    choices=[0, 0, 0, 0, 0, 0, 1, 0],   # index 6 -> GLIDE
    epoch=30,
    lr=0.0001,
    load=None,                          # path to a checkpoint to resume from, or None
    image_root="../GenImage_root",
    isPatch=True,
    img_height=256,
    bit_mode="thresholding",
    patch_size=32,
    patch_mode="max",
    gpu_id="0",
    val_batchsize=64,
    unbiased=False,
)

config.isTrain = True
config.isVal = False

if config.unbiased:
    config.qf = 96

unbiased_suffix = '_unbiased' if config.unbiased else ''
qf_subdir = f'qf{config.qf}/' if config.unbiased else ''
active_subsets = [MODEL_NAME_MAP[i] for i, flag in enumerate(config.choices) if flag]
subset_subdir = '+'.join(active_subsets) if active_subsets else 'none'
config.save_path = (
    f'../weights{unbiased_suffix}/{subset_subdir}/'
    f'{config.bit_mode}/{config.patch_mode}/{qf_subdir}'
)

val_config = types.SimpleNamespace(**vars(config))
val_config.isTrain = False
val_config.isVal = True

os.makedirs(config.save_path, exist_ok=True)
print("save_path:", config.save_path)
for k, v in sorted(vars(config).items()):
    print(f"  {k}: {v}")

## Preprocessing / data loading

Builds the train/val `DataLoader`s. The preprocessing itself (bit-mode transform, patch extraction via `bit_patch`, `ToTensor`, ImageNet normalization) is wired up inside `get_loader`/`get_val_loader` -> `create_preprocessing_pipeline` (see `loader.py`); this cell is where it's actually invoked (dataset construction + first batch pull below trigger it).

In [ ]:
train_loader = fetch_train_data(config)
val_loader = fetch_val_data(val_config)

print(f"Train batches per epoch: {len(train_loader)} (batchsize={config.batchsize})")
for entry in val_loader:
    print(f"Val set: {entry['name']:<24} nature={entry['nature_size']:<6} ai={entry['ai_size']}")

### (optional) Visualize a preprocessed sample

Pulls one training batch and un-normalizes the first image, so you can see exactly what the model receives: the upsampled 32x32 patch selected by `bit_patch` (see prior discussion — `bit_mode='thresholding'` is currently a no-op passthrough).

In [ ]:
import matplotlib.pyplot as plt

sample_inputs, sample_labels = next(iter(train_loader))

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img = (sample_inputs[0] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

plt.imshow(img)
plt.title(f"label={sample_labels[0].item():.0f} (1=natural, 0=AI)")
plt.axis("off")
plt.show()

## Model & optimizer

In [ ]:
model_net = NeuralNetwork().cuda()

if config.load:
    model_net.load_state_dict(torch.load(config.load))
    print(f"Loaded model from {config.load}")

optimizer = torch.optim.Adam(model_net.parameters(), config.lr)

best_performing_epoch = 0
highest_accuracy = 0

config_path = toolkit.save_config(config, config.save_path)
print(f"Saved config to {config_path}")

## Train one epoch

Re-run this cell to train the next epoch (it keeps an `current_epoch` counter across reruns and applies the same `poly_lr` schedule `train.py` uses). Restart the kernel / rerun the model cell above to start over from epoch 1.

In [ ]:
def _train_one_epoch(epoch_index):
    model_net.train()
    total_loss = 0.0
    epoch_iterations = 0

    current_lr = toolkit.poly_lr(optimizer, config.lr, epoch_index, config.epoch)
    print(f"Epoch {epoch_index} | lr={current_lr:.6f}")

    for batch_idx, (inputs, targets) in enumerate(train_loader, start=1):
        optimizer.zero_grad()

        inputs = inputs.cuda()
        targets = targets.cuda()

        outputs = model_net(inputs).ravel()

        loss_function = compute_binary_loss()
        batch_loss = loss_function(outputs, targets)

        batch_loss.backward()
        optimizer.step()

        epoch_iterations += 1
        total_loss += batch_loss.item()

        if batch_idx % 500 == 0 or batch_idx == len(train_loader) or batch_idx == 1:
            print(f"  iter {batch_idx:04d}/{len(train_loader):04d} | loss={batch_loss.item():.6f}")

    avg_loss = total_loss / max(epoch_iterations, 1)
    print(f"Epoch {epoch_index} done | avg_loss={avg_loss:.6f}")

    if epoch_index % 50 == 0:
        checkpoint_path = os.path.join(config.save_path, f'Network_epoch_{epoch_index}.pth')
        torch.save(model_net.state_dict(), checkpoint_path)
        print(f"Saved checkpoint to {checkpoint_path}")

    return avg_loss


if 'current_epoch' not in globals():
    current_epoch = 0
current_epoch += 1

avg_loss = _train_one_epoch(current_epoch)

## (optional) Validate current epoch

Mirrors `perform_validation` in `train.py`: evaluates on every selected dataset, reports accuracy, and saves `Network_best.pth` into `config.save_path` when accuracy improves.

In [ ]:
def _validate(epoch_index):
    global best_performing_epoch, highest_accuracy
    model_net.eval()

    total_correct = total_samples = 0

    with torch.no_grad():
        for dataset in val_loader:
            correct_ai = correct_nature = 0

            for inputs, targets in dataset['val_ai_loader']:
                inputs, targets = inputs.cuda(), targets.cuda()
                probabilities = torch.sigmoid(model_net(inputs)).ravel()
                correct = ((probabilities > 0.5) & (targets == 1)) | ((probabilities < 0.5) & (targets == 0))
                correct_ai += correct.sum().item()

            for inputs, targets in dataset['val_nature_loader']:
                inputs, targets = inputs.cuda(), targets.cuda()
                probabilities = torch.sigmoid(model_net(inputs)).ravel()
                correct = ((probabilities > 0.5) & (targets == 1)) | ((probabilities < 0.5) & (targets == 0))
                correct_nature += correct.sum().item()

            ai_count, nature_count = dataset['ai_size'], dataset['nature_size']
            dataset_acc = (correct_ai + correct_nature) / (ai_count + nature_count)
            total_correct += correct_ai + correct_nature
            total_samples += ai_count + nature_count
            print(f"  {dataset['name']:<24} accuracy={dataset_acc:.4f}")

    overall_accuracy = total_correct / total_samples

    if epoch_index == 1 or overall_accuracy > highest_accuracy:
        best_performing_epoch = epoch_index
        highest_accuracy = overall_accuracy
        best_model_path = os.path.join(config.save_path, 'Network_best.pth')
        torch.save(model_net.state_dict(), best_model_path)
        print(f"Saved best model ({best_model_path}) on epoch {epoch_index}")

    print(f"Epoch {epoch_index} | overall_accuracy={overall_accuracy:.4f} | best={highest_accuracy:.4f} (epoch {best_performing_epoch})")
    return overall_accuracy


val_accuracy = _validate(current_epoch)